# Yanolja 리뷰 크롤링 및 분석

이번 노트북에서는 Selenium을 사용하여 Yanolja의 호텔 리뷰 페이지에서 데이터를 크롤링하고, 수집한 데이터에 대해 분석을 진행합니다. 이 과정에서는 웹페이지 로드, 데이터 추출, 텍스트 처리 및 분석 결과를 Excel 파일로 저장하는 작업을 포함합니다.

### 1단계: Selenium으로 웹페이지 로드

Selenium을 사용하여 Yanolja 리뷰 페이지를 로드하고, 스크롤을 내려서 더 많은 데이터를 가져옵니다.

In [ ]:
!pip install selenium
!pip install bs4
!pip install pandas
!pip install openpyxl

In [1]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

# Selenium 드라이버 설정 (Chrome 사용)
driver = webdriver.Chrome() 

# Yanolja 리뷰 페이지로 이동
url = 'https://www.yanolja.com/reviews/domestic/10041505'
######## your code here ########
driver.get(url)

# 페이지 로딩을 위해 대기
time.sleep(3)

# 스크롤 설정: 페이지 하단까지 스크롤을 내리기
scroll_count = 10  # 스크롤 횟수 설정
for _ in range(scroll_count):
    ######## your code here ########
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)  # 스크롤 이후 대기

### 2단계: 페이지 소스 가져오기
웹페이지의 HTML 소스를 가져와서 BeautifulSoup을 사용해 데이터를 파싱합니다.

In [2]:
from bs4 import BeautifulSoup

# 웹페이지 소스 가져오기
page_source = driver.page_source

# BeautifulSoup를 사용하여 HTML 파싱
soup = BeautifulSoup(page_source, 'html.parser')

### 3단계: 리뷰 텍스트 추출
리뷰 텍스트를 추출하고 불필요한 공백이나 줄 바꿈을 제거합니다.

In [3]:
# 리뷰 텍스트 추출
################################
reviews_class = soup.select(".content-text.css-vjs6b8")
################################
reviews = []

# 각 리뷰 텍스트 정리 후 추가
for review in reviews_class:
    cleaned_text = review.get_text(strip=True).replace('\r', '').replace('\n', '')
    reviews.append(cleaned_text)

reviews

['친구들 3명에서 급하게 예약을 했던 숙손데 와.. 우선 프론트 직원님들.. 진짜 너무 친절해서 기분이 좋았습니다. 2박3일 일정이였는데 기분좋게 하루를 시작했고 숙소 위치도 그냥 완전 좋아요..!!!!! 애월은 마스터했습니다. ㅋㅋㅋㅋ 위치 금액 시설이용 서비스 위생 뭐 하나 빠지는거없이  (친구들도인정) 너무 편하게 잘 쉬다가 갑니다 ㅎㅎㅎ 후기 잘 안쓰는데 여기는 꼭 쓰고싶어서 이렇게 남깁니다!오늘도 좋은 하루 되세용:)🍀',
 '곽지해수욕장 앞이라 너무 좋습니다.곽지에서 한담해변 올레길 강추드리구요 아침에도 해질녘에도 너무 좋습니다',
 '너무 너무 좋았어요. 무엇보다 밝은 얼굴로 환대해주신 지원분들, 칭찬합니다. 비치도 가깝고, 룸은 작지만 깨끗하고 불편함 없었습니다.',
 '진짜 어지간해서는 후기 절대 안남기는데 사장님이신지 직원분이신지 친절함에 너무너무 감동받아서 남겨요 ..💓놀러다니는거 좋아해서 여러지역 숙박업소 다 가봤는데 여기 직원분들만큼 친절한 곳은 본 적이 없습니다 🥺 입실 몇시간 전부터 미리 전화 주시고 되게 세심하게 챙겨주셨어요 ! 또 방도 생각보다 넓고 뷰도 너무 이쁨요 지짜..👍🏻👍🏻 너무 과한 친절은 불편할 수 있는데 절대절대 그렇게 느껴지지도 않았고 삼일내내 기분 좋았어요 !! 처음에 1박 예약했다가 그냥 후기 좋길래 이틀 다 여기서 지내자 하고 연박 잡았는데 직원분들 덕분에 2박 3일 진짜 즐겁고 재밌게 지내다 갑니다 ! 너무 감사드려요❤️',
 '작년에 친구들이랑 방문 했다가 좋아서 다른 친구 데리고 다시 방문 했는데 머리 묶으신 여자 직원분 여전히 진짜 너무 친절하시고 좋아요주차 공간이 많지 않은 거 같길래 밤에 차 빼면 차 댈 데 없으려나 하고 걱정했는데 11시에도 지하 주차장 자리 많았어요 보니까 지상에도 자리가 있더라구요지하 1층에는 코인 세탁 건조기도 있고 편의점 가깝고 바다 가깝고 외부에 모래 씻어내는 용도로 있는 샤워기에도 필터까지 끼워져 있는 거 보고 세심한 거까지 신경쓰신다 느꼈어요 ㄴ담에 제주 가면 또 가

### 4단계: 별점 데이터 추출
HTML에서 별점 데이터를 추출하고, 각 리뷰의 별점을 계산합니다.

In [4]:
# 별점 추출
ratings = []
################################
rating_containers = soup.select(".css-rz7kwu")
################################

# 각 리뷰별로 별점 계산
for container in rating_containers:
    
    stars=container.select("svg path")
    
    rating = 0

    for star in stars:
        if not star.has_attr("fill-rule"):
            rating +=1
    
    ratings.append(rating)
    
ratings

[5,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 3,
 5,
 5,
 5,
 5,
 5,
 4,
 4,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 4,
 4,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 3,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 4,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 4,
 5,
 5,
 4,
 4,
 5,
 5,
 5,
 4,
 4,
 4,
 5,
 5,
 4,
 3,
 4,
 5,
 4,
 3,
 4,
 4,
 5,
 3,
 5,
 5,
 2,
 5,
 1,
 4,
 3,
 3,
 4,
 5,
 5,
 5,
 3,
 1,
 3,
 5,
 3,
 1,
 2,
 3,
 1,
 2,
 1,
 1,
 1,
 3,
 5,
 5,
 5,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 1,
 5,
 5,
 5,
 5,
 1,
 5,
 5,
 5,
 5,
 3,
 4,
 4,
 5]

### 5단계: 데이터 정리 및 DataFrame으로 변환
수집된 데이터를 Pandas DataFrame으로 변환하여 후속 분석을 용이하게 만듭니다.

In [5]:
import pandas as pd

# 별점과 리뷰를 결합하여 리스트 생성
data = list(zip(ratings, reviews))

# DataFrame으로 변환
df_reviews = pd.DataFrame(data, columns=['Rating', 'Review'])
df_reviews

,Rating,Review
0,5,친구들 3명에서 급하게 예약을 했던 숙손데 와.. 우선 프론트 직원님들.. 진짜 너...
1,5,곽지해수욕장 앞이라 너무 좋습니다.곽지에서 한담해변 올레길 강추드리구요 아침에도 해...
2,5,"너무 너무 좋았어요. 무엇보다 밝은 얼굴로 환대해주신 지원분들, 칭찬합니다. 비치도..."
3,5,진짜 어지간해서는 후기 절대 안남기는데 사장님이신지 직원분이신지 친절함에 너무너무 ...
4,5,작년에 친구들이랑 방문 했다가 좋아서 다른 친구 데리고 다시 방문 했는데 머리 묶으...
...,...,...
215,5,깔끔하고 좋았습니다... 단점은 날씨가 너무 안좋았다는점....
216,3,직원분들은 친절하시지만 추가 물품(수건 물)은 추가비용내야하는건 아쉬움
217,4,감사합니다감사합니다
218,4,비다뷰가 압권입니다. 연식이 있어 좀 적극적인 관리가 필요해 보입니다


### 6단계: 리뷰 분석 - 평균 별점 계산
수집된 리뷰에서 평균 별점을 계산합니다.

In [6]:
# 평균 별점 계산
ratings=0

for rating in df_reviews['Rating']:
    ratings+=rating

average_rating = ratings / len(df_reviews)

average_rating

4.509090909090909

### 7단계: 자주 등장하는 단어 추출
리뷰 텍스트에서 자주 등장하는 단어를 추출하고, 불용어를 제거하여 분석합니다.

In [8]:
from collections import Counter
import re

# 불용어 리스트 (한국어)
korean_stopwords = set(['이', '그', '저', '것', '들', '다', '을', '를', '에', '의', '가', '이', '는', '해', '한', '하', '하고', '에서', '에게', '과', '와', '너무', '잘', '또','좀', '호텔', '아주', '진짜', '정말'])

# 모든 리뷰를 하나의 문자열로 결합
all_reviews_text = " ".join(df_reviews['Review'])

# 단어 추출 (특수문자 제거)
words = re.findall(r'[가-힣]+',all_reviews_text)

# 불용어 제거
filtered_words = []

for word in words:
    if word not in korean_stopwords:
        filtered_words.append(word)

# 단어 빈도 계산
word_counts = Counter(filtered_words)

# 자주 등장하는 상위 15개 단어 추출
common_words = word_counts.most_common(15)

common_words

[('좋고', 38),
 ('좋았습니다', 36),
 ('바로', 36),
 ('좋았어요', 33),
 ('있어서', 33),
 ('좋아요', 29),
 ('친절하시고', 21),
 ('직원분들', 19),
 ('객실', 19),
 ('다시', 17),
 ('곽지해수욕장', 16),
 ('뷰가', 16),
 ('깨끗하고', 15),
 ('수', 14),
 ('깔끔하고', 14)]

### 8단계: 분석 결과 요약
평균 별점과 자주 등장하는 단어를 DataFrame으로 만들어 최종 분석 결과를 요약합니다.

In [9]:
# 분석 결과 요약
summary_df = pd.DataFrame({
    'Average Rating': [average_rating],
    'Common Words': [', '.join([f"{word}({count})" for word, count in common_words])]
})

# 최종 DataFrame 결합
final_df = pd.concat([df_reviews, summary_df], ignore_index=True)
final_df

,Rating,Review,Average Rating,Common Words
0,5.0,친구들 3명에서 급하게 예약을 했던 숙손데 와.. 우선 프론트 직원님들.. 진짜 너...,NaN,NaN
1,5.0,곽지해수욕장 앞이라 너무 좋습니다.곽지에서 한담해변 올레길 강추드리구요 아침에도 해...,NaN,NaN
2,5.0,"너무 너무 좋았어요. 무엇보다 밝은 얼굴로 환대해주신 지원분들, 칭찬합니다. 비치도...",NaN,NaN
3,5.0,진짜 어지간해서는 후기 절대 안남기는데 사장님이신지 직원분이신지 친절함에 너무너무 ...,NaN,NaN
4,5.0,작년에 친구들이랑 방문 했다가 좋아서 다른 친구 데리고 다시 방문 했는데 머리 묶으...,NaN,NaN
...,...,...,...,...
216,3.0,직원분들은 친절하시지만 추가 물품(수건 물)은 추가비용내야하는건 아쉬움,NaN,NaN
217,4.0,감사합니다감사합니다,NaN,NaN
218,4.0,비다뷰가 압권입니다. 연식이 있어 좀 적극적인 관리가 필요해 보입니다,NaN,NaN
219,5.0,잘 쉬다가 가요 바다뷰 좋아요,NaN,NaN


### 9단계: Excel 파일로 저장
최종 결과를 Excel 파일로 저장합니다.

In [11]:
# Excel 파일로 저장
######## your code here ########
final_df.to_excel("yanolija.xlsx",index=[False])

### 10단계: 드라이버 종료
크롤링이 끝난 후, Selenium 드라이버를 종료합니다.

In [12]:
# 드라이버 종료
driver.quit()